In [1]:
import boto3, bz2
import pandas as pd

s3 = boto3.client("s3")
bucket = "sagemaker-us-east-1-926705662061"
prefix = "Amazon-ML-Hackathon/"

s3.download_file(bucket, prefix + "test.ft.txt.bz2", "test.ft.txt.bz2")
s3.download_file(bucket, prefix + "train.ft.txt.bz2", "train.ft.txt.bz2")

def load_ft_bz2(path, limit=None):
    labels, texts = [], []
    with bz2.open(path, mode="rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            label, text = line.split(" ", 1)
            labels.append(int(label.replace("__label__", "")))
            texts.append(text.strip())
    return pd.DataFrame({"label": labels, "text": texts})

train_df = load_ft_bz2("train.ft.txt.bz2", limit=150000)
test_df = load_ft_bz2("test.ft.txt.bz2", limit=50000)

print(train_df.shape, test_df.shape)
train_df['label'].value_counts()

(150000, 2) (50000, 2)


label
2    76248
1    73752
Name: count, dtype: int64

In [2]:
import re
import numpy as np

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

train_df['rating'] = train_df['label'].map({1: 1, 2: 5})
test_df['rating'] = test_df['label'].map({1: 1, 2: 5})

for df in [train_df, test_df]:
    df['char_count'] = df['clean_text'].str.len()
    df['word_count'] = df['clean_text'].str.split().apply(len)
    df['sentence_count'] = df['text'].apply(lambda t: len(re.split(r'[.!?]+', t)))

dupes = train_df.duplicated(subset='clean_text').sum()
print("Duplicate cleaned texts in train:", dupes)

print("Empty cleaned texts:", (train_df['clean_text'].str.len() == 0).sum())

print(train_df['rating'].value_counts())
print(train_df[['char_count', 'word_count', 'sentence_count']].describe())

train_df.head()

Duplicate cleaned texts in train: 30
Empty cleaned texts: 0
rating
5    76248
1    73752
Name: count, dtype: int64
          char_count     word_count  sentence_count
count  150000.000000  150000.000000   150000.000000
mean      427.629093      82.406687        6.759273
std       234.428059      44.747759        3.221053
min        19.000000       4.000000        1.000000
25%       229.000000      45.000000        4.000000
50%       382.000000      74.000000        6.000000
75%       592.000000     114.000000        9.000000
max      1009.000000     242.000000       49.000000


,label,text,clean_text,rating,char_count,word_count,sentence_count
0,2,Stuning even for the non-gamer: This sound tra...,stuning even for the non gamer this sound trac...,5,415,80,7
1,2,The best soundtrack ever to anything.: I'm rea...,the best soundtrack ever to anything i m readi...,5,500,102,6
2,2,Amazing!: This soundtrack is my favorite music...,amazing this soundtrack is my favorite music o...,5,727,136,7
3,2,Excellent Soundtrack: I truly like this soundt...,excellent soundtrack i truly like this soundtr...,5,714,122,8
4,2,"Remember, Pull Your Jaw Off The Floor After He...",remember pull your jaw off the floor after hea...,5,462,90,6


In [3]:
dupes = train_df.duplicated(subset='clean_text').sum()
print("Duplicate cleaned texts in train:", dupes)
print("Empty cleaned texts:", (train_df['clean_text'].str.len() == 0).sum())
print(train_df['rating'].value_counts())
print(train_df[['char_count', 'word_count', 'sentence_count']].describe())

Duplicate cleaned texts in train: 30
Empty cleaned texts: 0
rating
5    76248
1    73752
Name: count, dtype: int64
          char_count     word_count  sentence_count
count  150000.000000  150000.000000   150000.000000
mean      427.629093      82.406687        6.759273
std       234.428059      44.747759        3.221053
min        19.000000       4.000000        1.000000
25%       229.000000      45.000000        4.000000
50%       382.000000      74.000000        6.000000
75%       592.000000     114.000000        9.000000
max      1009.000000     242.000000       49.000000


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

train_df = train_df.drop_duplicates(subset='clean_text').reset_index(drop=True)
print("Train shape after dedup:", train_df.shape)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    max_features=10000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

print("TF-IDF shape:", X_train_tfidf.shape, X_test_tfidf.shape)

num_cols = ['char_count', 'word_count', 'sentence_count']
scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_df[num_cols])
X_test_num = scaler.transform(test_df[num_cols])

X_train = hstack([X_train_tfidf, X_train_num])
X_test = hstack([X_test_tfidf, X_test_num])

y_train = train_df['rating']
y_test = test_df['rating']

print("Final combined shapes:", X_train.shape, X_test.shape)

Train shape after dedup: (149970, 7)
TF-IDF shape: (149970, 10000) (50000, 10000)
Final combined shapes: (149970, 10003) (50000, 10003)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print("Train:", X_tr.shape, "Val:", X_val.shape)

results = {}

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_tr, y_tr)
lr_preds = lr.predict(X_val)
results['Logistic Regression'] = f1_score(y_val, lr_preds, average='macro')
print("Logistic Regression Macro-F1:", results['Logistic Regression'])

svm = LinearSVC(class_weight='balanced', random_state=42)
svm.fit(X_tr, y_tr)
svm_preds = svm.predict(X_val)
results['Linear SVM'] = f1_score(y_val, svm_preds, average='macro')
print("Linear SVM Macro-F1:", results['Linear SVM'])

lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    max_bin=63,
    force_col_wise=True,
    num_leaves=31,
    n_jobs=2
)
lgbm.fit(X_tr, y_tr)
lgbm_preds = lgbm.predict(X_val)
results['LightGBM'] = f1_score(y_val, lgbm_preds, average='macro')
print("LightGBM Macro-F1:", results['LightGBM'])

print("\n=== Macro-F1 Comparison ===")
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name}: {score:.4f}")

Train: (119976, 10003) Val: (29994, 10003)
Logistic Regression Macro-F1: 0.9133572322778873
Linear SVM Macro-F1: 0.9103184716407047
[LightGBM] [Info] Number of positive: 60991, number of negative: 58985
[LightGBM] [Info] Total Bins 612484
[LightGBM] [Info] Number of data points in the train set: 119976, number of used features: 10003
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/opt/conda/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Macro-F1: 0.90004173122336

=== Macro-F1 Comparison ===
Logistic Regression: 0.9134
Linear SVM: 0.9103
LightGBM: 0.9000


In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1
)
grid.fit(X_tr, y_tr)

print("Best params:", grid.best_params_)
print("Best CV Macro-F1:", grid.best_score_)

best_lr = grid.best_estimator_
val_preds = best_lr.predict(X_val)
print("Validation Macro-F1:", f1_score(y_val, val_preds, average='macro'))

Best params: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Macro-F1: 0.9088700878424074
Validation Macro-F1: 0.9133572322778873


In [7]:
test_df.to_csv('test_df_full.csv', index=False)
print(test_df.shape)
print(test_df.head())

(50000, 7)
   label                                               text  \
0      2  Great CD: My lovely Pat has one of the GREAT v...   
1      2  One of the best game music soundtracks - for a...   
2      1  Batteries died within a year ...: I bought thi...   
3      2  works fine, but Maha Energy is better: Check o...   
4      2  Great for the non-audiophile: Reviewed quite a...   

                                          clean_text  rating  char_count  \
0  great cd my lovely pat has one of the great vo...       5         513   
1  one of the best game music soundtracks for a g...       5         798   
2  batteries died within a year i bought this cha...       1         323   
3  works fine but maha energy is better check out...       5         221   
4  great for the non audiophile reviewed quite a ...       5         403   

   word_count  sentence_count  
0         107              11  
1         152               7  
2          59               6  
3          39            

In [8]:
test_preds = best_lr.predict(X_test)

export_df = test_df.copy()
export_df['predicted_rating'] = test_preds
export_df.to_csv('test_predictions_full.csv', index=False)

print(export_df.shape)
print(export_df.head())

(50000, 8)
   label                                               text  \
0      2  Great CD: My lovely Pat has one of the GREAT v...   
1      2  One of the best game music soundtracks - for a...   
2      1  Batteries died within a year ...: I bought thi...   
3      2  works fine, but Maha Energy is better: Check o...   
4      2  Great for the non-audiophile: Reviewed quite a...   

                                          clean_text  rating  char_count  \
0  great cd my lovely pat has one of the great vo...       5         513   
1  one of the best game music soundtracks for a g...       5         798   
2  batteries died within a year i bought this cha...       1         323   
3  works fine but maha energy is better check out...       5         221   
4  great for the non audiophile reviewed quite a ...       5         403   

   word_count  sentence_count  predicted_rating  
0         107              11                 5  
1         152               7                 5  
2  